In [75]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report
import joblib

import warnings
warnings.filterwarnings('ignore')

In [12]:
df = pd.read_csv(r'../data/processed/model_features.csv', parse_dates=['date'])
df.head()

,neutral,home_rank,home_points,away_rank,away_points,rank_diff,tournament_bucket_continental_championship,tournament_bucket_friendly,tournament_bucket_other,tournament_bucket_world_cup,...,home_form_win_rate,home_form_goal_for,home_form_goal_against,away_form_win_rate,away_form_goal_for,away_form_goal_against,date,home_team,away_team,result_encoded
0,True,53.0,560.0,13.0,687.0,-40.0,1.0,0.0,0.0,0.0,...,1.00,3.00,1.0,0.6,1.6,1.0,2001-07-18,Peru,Mexico,2
1,True,69.0,516.0,33.0,610.0,-36.0,1.0,0.0,0.0,0.0,...,1.00,2.00,0.0,0.4,1.0,1.0,2001-07-19,Bolivia,Costa Rica,0
2,True,48.0,578.0,30.0,620.0,-18.0,1.0,0.0,0.0,0.0,...,0.67,1.33,2.0,0.6,1.8,1.0,2001-07-19,Honduras,Uruguay,2
3,False,109.0,404.0,29.0,621.0,-80.0,0.0,1.0,0.0,0.0,...,0.75,1.25,1.0,0.4,1.4,1.0,2001-07-19,Malaysia,Saudi Arabia,0
4,False,82.0,483.0,45.0,584.0,-37.0,0.0,1.0,0.0,0.0,...,0.60,1.00,2.0,0.4,1.8,1.0,2001-07-20,Cuba,Jamaica,1


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20929 entries, 0 to 20928
Data columns (total 21 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   neutral                                     20929 non-null  bool          
 1   home_rank                                   20929 non-null  float64       
 2   home_points                                 20929 non-null  float64       
 3   away_rank                                   20929 non-null  float64       
 4   away_points                                 20929 non-null  float64       
 5   rank_diff                                   20929 non-null  float64       
 6   tournament_bucket_continental_championship  20929 non-null  float64       
 7   tournament_bucket_friendly                  20929 non-null  float64       
 8   tournament_bucket_other                     20929 non-null  float64       
 9   tournament_bucket

In [15]:
train = df[df['date'] <= '2025-01-1']
test = df[df['date'] > '2025-01-1']

In [18]:
train.tail()

,neutral,home_rank,home_points,away_rank,away_points,rank_diff,tournament_bucket_continental_championship,tournament_bucket_friendly,tournament_bucket_other,tournament_bucket_world_cup,...,home_form_win_rate,home_form_goal_for,home_form_goal_against,away_form_win_rate,away_form_goal_for,away_form_goal_against,date,home_team,away_team,result_encoded
19644,True,81.0,1302.86,155.0,1021.24,74.0,0.0,0.0,1.0,0.0,...,0.2,1.0,2.0,0.4,1.6,1.0,2024-12-28,Bahrain,Yemen,0
19645,False,116.0,1168.02,159.0,1008.26,43.0,0.0,0.0,1.0,0.0,...,0.2,1.2,2.0,0.4,1.6,1.0,2024-12-29,Vietnam,Singapore,2
19646,False,100.0,1218.56,147.0,1053.03,47.0,0.0,0.0,1.0,0.0,...,0.2,1.4,2.0,0.4,1.6,1.0,2024-12-30,Thailand,Philippines,2
19647,True,76.0,1326.18,56.0,1431.30,-20.0,0.0,0.0,1.0,0.0,...,0.4,1.8,2.0,0.4,1.6,2.0,2024-12-31,Oman,Saudi Arabia,2
19648,False,137.0,1098.42,81.0,1302.86,-56.0,0.0,0.0,1.0,0.0,...,0.6,2.0,2.0,0.4,1.6,2.0,2024-12-31,Kuwait,Bahrain,0


In [17]:
test.head()

,neutral,home_rank,home_points,away_rank,away_points,rank_diff,tournament_bucket_continental_championship,tournament_bucket_friendly,tournament_bucket_other,tournament_bucket_world_cup,...,home_form_win_rate,home_form_goal_for,home_form_goal_against,away_form_win_rate,away_form_goal_for,away_form_goal_against,date,home_team,away_team,result_encoded
19649,False,116.0,1168.02,100.0,1218.56,-16.0,0.0,0.0,1.0,0.0,...,0.6,1.8,1.0,0.4,1.2,2.0,2025-01-02,Vietnam,Thailand,2
19650,True,67.0,1375.16,108.0,1195.45,41.0,0.0,0.0,1.0,0.0,...,0.8,2.0,1.0,0.2,1.0,2.0,2025-01-04,Oman,Bahrain,0
19651,True,76.0,1326.18,81.0,1302.86,5.0,0.0,0.0,1.0,0.0,...,0.6,1.6,1.0,0.4,1.2,2.0,2025-01-04,Burkina Faso,Kenya,1
19652,False,100.0,1218.56,116.0,1168.02,16.0,0.0,0.0,1.0,0.0,...,0.4,1.2,1.0,0.4,1.2,1.0,2025-01-05,Thailand,Vietnam,0
19653,True,114.0,1174.99,108.0,1195.45,-6.0,0.0,0.0,1.0,0.0,...,0.2,1.2,2.0,0.6,1.6,1.0,2025-01-07,Tanzania,Kenya,0


In [64]:
feature_cols = ['neutral', 'home_rank', 'home_points', 'away_rank', 'away_points',
       'rank_diff', 'tournament_bucket_continental_championship',
       'tournament_bucket_friendly', 'tournament_bucket_other',
       'tournament_bucket_world_cup', 'tournament_bucket_world_cup_qualifier',
       'home_form_win_rate', 'home_form_goal_for', 'home_form_goal_against',
       'away_form_win_rate', 'away_form_goal_for', 'away_form_goal_against',]
target = ['result_encoded']

X_train = train[feature_cols]

y_train = train[target]

X_test = test[feature_cols]

y_test = test[target]

In [65]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(19649, 17)
(19649, 1)
(1280, 17)
(1280, 1)


In [66]:
# cheking if the model will predict draws or not
naive_preds = np.where(X_test["rank_diff"] > 0, 2, 0)  # 2=Home, 0=Away
naive_accuracy = accuracy_score(y_test, naive_preds)
print(f"Naive baseline accuracy: {naive_accuracy:.4f}")

Naive baseline accuracy: 0.4344


In [67]:
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [68]:
log_reg_pred = log_reg.predict(X_test)
log_reg_proba = log_reg.predict_proba(X_test)


print(f"Logistic Regression accuracy: {accuracy_score(y_test, log_reg_pred):.4f}")
print(f"Logistic Regression log-loss: {log_loss(y_test, log_reg_proba):.4f}")
print()
print(classification_report(y_test, log_reg_pred, target_names=["Away", "Draw", "Home"]))


Logistic Regression accuracy: 0.4844
Logistic Regression log-loss: 1.0435

              precision    recall  f1-score   support

        Away       0.37      0.02      0.04       378
        Draw       0.00      0.00      0.00       280
        Home       0.49      0.99      0.65       622

    accuracy                           0.48      1280
   macro avg       0.28      0.33      0.23      1280
weighted avg       0.35      0.48      0.33      1280



In [69]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
)


In [70]:
model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegresso

In [71]:
xgb_pred = model.predict(X_test)
xgb_proba = model.predict_proba(X_test)
accuracy = accuracy_score(xgb_pred, y_test)

print(f"XGBoost accuracy: {accuracy_score(y_test, xgb_pred):.4f}")
print(f"XGBoost log-loss: {log_loss(y_test, xgb_proba):.4f}")
print()
print(classification_report(y_test, xgb_pred, target_names=["Away", "Draw", "Home"]))

XGBoost accuracy: 0.5070
XGBoost log-loss: 1.0380

              precision    recall  f1-score   support

        Away       0.48      0.21      0.29       378
        Draw       0.33      0.01      0.02       280
        Home       0.51      0.91      0.66       622

    accuracy                           0.51      1280
   macro avg       0.44      0.38      0.32      1280
weighted avg       0.46      0.51      0.41      1280



In [74]:
importance_df = pd.DataFrame({
    'features' : feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

importance_df

,features,importance
0,neutral,0.208597
7,tournament_bucket_friendly,0.097925
5,rank_diff,0.077452
10,tournament_bucket_world_cup_qualifier,0.052620
8,tournament_bucket_other,0.051439
13,home_form_goal_against,0.050371
2,home_points,0.044887
9,tournament_bucket_world_cup,0.044713
1,home_rank,0.044580
4,away_points,0.044389


In [76]:
model_path = "../models/xgboost_v1.pkl"
joblib.dump(model, model_path)
print(f"Saved model to {model_path}")

Saved model to ../models/xgboost_v1.pkl


In [77]:
joblib.dump(feature_cols, "../models/feature_columns.pkl")

['../models/feature_columns.pkl']